# Solutions 1 & 2 — Encoder comparison + rule-based mitigation

**No API calls.** Everything runs offline on free Colab.

**Solution 1 (encoder):** Step 1 showed retrieval getting *worse* as dense weighting rose (R@5 0.767 at α=0.3 → 0.367 at α=1.0), meaning `multilingual-MiniLM` is actively harmful on Arabic. This compares it against multilingual-e5, mpnet, DarijaBERT and XLM-RoBERTa-Morocco.

**Solution 2 (rule-based mitigation):** deterministic dialect→MSA function-word substitution, plus a rule-based expansion variant. Zero cost, fully reproducible, and gives the paper a no-LLM baseline.

**Upload:** `corpus_v2.json`. **No API key needed.** Use a GPU runtime if available (Runtime → Change runtime type → T4) — the BERT models are much faster on GPU.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "repo_raw": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main",
    "encoders": [
        # (huggingface id, needs e5-style prefixes, pooling)
        ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", False, "st"),
        ("intfloat/multilingual-e5-base",                               True,  "st"),
        ("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", False, "st"),
        # Raw BERT models: not sentence-transformers, so mean-pool manually.
        ("SI2M-Lab/DarijaBERT",                                          False, "mean"),
        ("atlasia/XLM-RoBERTa-Morocco",                                  False, "mean"),
    ],
    "alphas": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "k_values": (1, 3, 5, 10),
    "max_k": 10,
}

### Load corpus and QA pairs

In [ ]:
import json, requests

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
qa_pairs = json.loads(requests.get(f"{CONFIG['repo_raw']}/data/qa_pairs.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
missing = {q["source_chunk_id"] for q in qa_pairs} - set(corpus_ids)

print(f"Corpus: {len(corpus)} passages | QA: {len(qa_pairs)} | unresolved gold: {len(missing)}")

### Arabic normalization + BM25 (shared across all encoders)

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
print("BM25 index built.")

### SOLUTION 2: rule-based Darija -> MSA (no API)

In [ ]:
# Converts dialect function words only; every content word is left untouched.
# This is deliberately the opposite of full LLM normalisation, which rewrote
# content words into synonyms and collapsed the ranking margin in Step 1.
DARIJA_TO_MSA = {
    "شنو هي": "ما هي", "شنو هو": "ما هو", "شنو هما": "ما هي", "شنو هوما": "ما هي",
    "شحال ديال": "كم", "بشحال": "بكم", "شحال": "كم",
    "فين": "أين", "لفين": "إلى أين", "منين": "من أين",
    "علاش": "لماذا", "علاه": "لماذا", "كيفاش": "كيف",
    "إمتى": "متى", "فوقاش": "متى", "شكون": "من", "أشمن": "أي",
    "فأش": "في أي", "فأي": "في أي", "بأش": "بماذا", "أش": "ماذا", "شنو": "ما",
    "واش": "هل", "كاينة": "توجد", "كاين": "يوجد", "ماكاينش": "لا يوجد",
    "بزاف": "كثيرا", "دابا": "الآن", "غادي": "سوف", "باش": "لكي",
    "هادشي": "هذا", "هادي": "هذه", "هاد": "هذا",
    "بحال": "مثل", "حيت": "لأن", "ملي": "عندما", "واخا": "رغم",
    "ماشي": "ليس", "بلا": "بدون", "ديالو": "", "ديالها": "", "ديالهم": "", "ديال": "",
}

def rule_normalize(text):
    out = text
    for dialect, msa in sorted(DARIJA_TO_MSA.items(), key=lambda kv: -len(kv[0])):
        out = re.sub(rf"(?<!\w){re.escape(dialect)}(?!\w)", msa, out)
    return re.sub(r"\s+", " ", out).strip()

# Query variants available without any API
for item in qa_pairs:
    item["M4_rulebased"] = rule_normalize(item["darija_query"])
    item["M3_rule_expansion"] = f'{item["darija_query"]} {item["M4_rulebased"]}'

print("Examples:")
for item in qa_pairs[:4]:
    print(f'  Darija : {item["darija_query"]}')
    print(f'  M4     : {item["M4_rulebased"]}')
    print()

### SOLUTION 1: encoder loading (handles both ST and raw BERT models)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

def encode_corpus(model_id, use_prefix, kind):
    """Returns (encode_query_fn, corpus_embedding_matrix)."""
    if kind == "st":
        model = SentenceTransformer(model_id)
        pfx_p = "passage: " if use_prefix else ""
        pfx_q = "query: " if use_prefix else ""
        emb = model.encode([pfx_p + t for t in corpus_texts],
                           normalize_embeddings=True, show_progress_bar=True, batch_size=64)
        return (lambda q: model.encode([pfx_q + q], normalize_embeddings=True)[0]), np.asarray(emb, dtype="float32")

    # Raw BERT: mean-pool the last hidden state over non-padding tokens.
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModel.from_pretrained(model_id)
    mdl.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mdl.to(device)

    def embed(texts, batch_size=32):
        vecs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
            with torch.no_grad():
                out = mdl(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            vecs.append(pooled.cpu().numpy())
        return np.vstack(vecs)

    emb = embed(corpus_texts)
    return (lambda q: embed([q])[0]), np.asarray(emb, dtype="float32")

### Retrieval + evaluation

In [ ]:
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def make_retriever(encode_query, corpus_emb):
    def retrieve(query, k, alpha):
        score = 0.0
        if alpha > 0:
            score = alpha * minmax(corpus_emb @ encode_query(query))
        if alpha < 1:
            score = score + (1 - alpha) * minmax(np.asarray(bm25.get_scores(tokenize(query))))
        idx = np.argsort(-score)[:k]
        return [corpus_ids[i] for i in idx]
    return retrieve

def evaluate(retrieve, field, alpha):
    hits = {k: 0 for k in CONFIG["k_values"]}
    rr = []
    for item in qa_pairs:
        got = retrieve(item[field], CONFIG["max_k"], alpha)
        gold = item["source_chunk_id"]
        for k in CONFIG["k_values"]:
            if gold in got[:k]:
                hits[k] += 1
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    n = len(qa_pairs)
    out = {f"R@{k}": hits[k] / n for k in CONFIG["k_values"]}
    out["MRR"] = sum(rr) / n
    return out

VARIANTS = ["msa_query", "darija_query", "M4_rulebased", "M3_rule_expansion"]

### Run the grid (encoders x alphas x variants)

In [ ]:
import pandas as pd, gc, traceback

rows = []
for model_id, use_prefix, kind in CONFIG["encoders"]:
    short = model_id.split("/")[-1]
    print(f"\n=== {short} ===")
    try:
        encode_query, corpus_emb = encode_corpus(model_id, use_prefix, kind)
    except Exception as e:
        # A model that fails to load shouldn't kill the whole grid.
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        continue

    retrieve = make_retriever(encode_query, corpus_emb)
    for alpha in CONFIG["alphas"]:
        for v in VARIANTS:
            rows.append({"encoder": short, "alpha": alpha, "variant": v,
                         **evaluate(retrieve, v, alpha)})
        print(f"  alpha={alpha} done")

    del encode_query, corpus_emb, retrieve
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

grid = pd.DataFrame(rows)
grid.to_csv("encoder_grid.csv", index=False)
print(f"\nGrid: {len(grid)} rows -> encoder_grid.csv")

### SOLUTION 1 verdict: which encoder handles Darija best?

In [ ]:
print("Best Darija (mismatch) R@5 per encoder, and the alpha that achieved it:\n")
dar = grid[grid.variant == "darija_query"]
best = dar.loc[dar.groupby("encoder")["R@5"].idxmax()][
    ["encoder", "alpha", "R@1", "R@5", "MRR"]
].sort_values("R@5", ascending=False)
print(best.to_string(index=False))

print("\nDoes dense retrieval still hurt? (Darija R@5 by alpha)\n")
pivot = dar.pivot_table(index="alpha", columns="encoder", values="R@5")
print(pivot.round(3).to_string())
print("\nIf a column RISES with alpha, that encoder is contributing usefully —")
print("unlike MiniLM in Step 1, where performance fell as alpha increased.")

### SOLUTION 2 verdict: does rule-based mitigation recover anything?

In [ ]:
print("Recovery by encoder (at each encoder's best alpha for Darija):\n")
out = []
for enc in grid.encoder.unique():
    sub_dar = dar[dar.encoder == enc]
    a = sub_dar.loc[sub_dar["R@5"].idxmax(), "alpha"]
    s = grid[(grid.encoder == enc) & (grid.alpha == a)].set_index("variant")
    base, mism = s.loc["msa_query", "R@5"], s.loc["darija_query", "R@5"]
    drop = base - mism
    for v in ["M4_rulebased", "M3_rule_expansion"]:
        val = s.loc[v, "R@5"]
        out.append({
            "encoder": enc, "alpha": a, "mitigation": v,
            "baseline": round(base, 3), "mismatch": round(mism, 3),
            "mitigated": round(val, 3),
            "recovery_%": round((val - mism) / drop * 100, 1) if drop > 0 else float("nan"),
        })

recovery = pd.DataFrame(out).sort_values("recovery_%", ascending=False)
print(recovery.to_string(index=False))
recovery.to_csv("recovery_no_api.csv", index=False)

top = recovery.iloc[0]
print("\n" + "=" * 70)
print(f"BEST SO FAR (no API used)")
print(f"  encoder    : {top['encoder']}  (alpha={top['alpha']})")
print(f"  mitigation : {top['mitigation']}")
print(f"  Recall@5   : {top['mismatch']} -> {top['mitigated']}   "
      f"(recovery {top['recovery_%']}%)")
print("=" * 70)

from google.colab import files
files.download("encoder_grid.csv")
files.download("recovery_no_api.csv")

# Next: if recovery is positive here, Solution 3 (LLM-based M2/M3) only needs
# to beat this number. If it is still negative, the problem is not the
# normaliser and the paper should be framed around the negative result.